# Parallel maximum entropy knockoffs

This notebook compares the existing `method=:maxent` solver with the adaptive window-parallel `method=:maxent_fast` solver. For an actual four-thread timing run, start Julia/Jupyter with `JULIA_NUM_THREADS=4` before opening the notebook.

In [1]:
using Knockoffs
using LinearAlgebra
using Random
using Statistics
using Distributions

BLAS.set_num_threads(1)
Threads.nthreads()

[ Info: Precompiling Knockoffs [878bf26d-0c49-448a-9df5-b057c815d613] (cache misses: include_dependency fsize change (2), wrong dep version loaded (2), dep missing source (4), mismatched flags (10))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


4

## Simulate AR(1) model-X data

In [2]:
Random.seed!(2026)

n = 500
p = 1000
k = 60
q = 0.10
rho = 0.4

Σ = [rho^abs(i - j) for i in 1:p, j in 1:p]
μ = zeros(p)
X = rand(MvNormal(μ, Symmetric(Σ)), n)' |> Matrix

causal = sort!(shuffle(1:p)[1:k])
β = zeros(p)
β[causal] .= rand([-1.0, 1.0], k) .* 2.5
y = X * β + randn(n);

In [6]:
function run_knockoff_trial(method::Symbol; solver_kwargs...)
    Random.seed!(2027)
    solve_time = @elapsed s = solve_s(Symmetric(Σ), method; verbose=true, solver_kwargs...)
    min_eig = eigmin(Symmetric(2Σ - Diagonal(s)))

    Random.seed!(2028)
    ko_time = @elapsed ko = modelX_gaussian_knockoffs(X, method, μ, Σ; solver_kwargs...)

    Random.seed!(2029)
    fit_time = @elapsed fit = fit_lasso(y, ko; filter_method=:knockoff, debias=nothing)
    selected = select_variables(fit, q)
    true_selected = intersect(selected, causal)
    false_selected = setdiff(selected, causal)
    fdp = isempty(selected) ? 0.0 : length(false_selected) / length(selected)
    power = length(true_selected) / length(causal)

    return (; method, solve_time, ko_time, fit_time, min_eig,
        selected=length(selected), fdp, power, mean_s=mean(s))
end

function print_result(r)
    println("method:     ", r.method)
    println("solve time: ", round(r.solve_time, digits=3), " seconds")
    println("ko time:    ", round(r.ko_time, digits=3), " seconds")
    println("fit time:   ", round(r.fit_time, digits=3), " seconds")
    println("min eig:    ", r.min_eig)
    println("selected:   ", r.selected)
    println("FDP:        ", round(r.fdp, digits=3))
    println("power:      ", round(r.power, digits=3))
    println("mean(s):    ", round(r.mean_s, digits=3))
end;

## Baseline: `method=:maxent`

In [10]:
maxent_result = run_knockoff_trial(:maxent; niter=100, tol=1e-5)
print_result(maxent_result)

Maxent initial obj = -700.7496748154989
Iter 1: obj = -592.9632484127432, δ = 0.3644471987139401
Iter 2: obj = -591.684732019377, δ = 0.03652973229743994
Iter 3: obj = -591.660109527087, δ = 0.004700979031753261
Iter 4: obj = -591.6596614125724, δ = 0.0006610575234006211
Iter 5: obj = -591.659653064711, δ = 8.819302816709396e-5
Iter 6: obj = -591.6596529271567, δ = 1.2030347537939079e-5
Iter 7: obj = -591.6596529225178, δ = 1.6264960241985094e-6
method:     maxent
solve time: 1.668 seconds
ko time:    1.79 seconds
fit time:   1.773 seconds
min eig:    0.24654543174649896
selected:   66
FDP:        0.091
power:      1.0
mean(s):    0.611


## Adaptive window-parallel: `method=:maxent_fast`

In [8]:
fast_result = run_knockoff_trial(:maxent_fast;
    niter=100,
    tol=1e-5,
    nworkers=4,
    feature_order=collect(1:p),
    boundary_band=25,
    min_window_size=200,
    window_corr_tol=0.5,
    factor_check_tol=1e-3)
print_result(fast_result)

Adaptive window-parallel setup:
  Feature ordering: user-supplied feature_order; original feature order was kept.
  Approximately independent windows: 4
  Julia threads available: 4
  Worker threads used: 4
  Boundary band: 25
  Minimum window size: 200
  Window correlation tolerance: 0.5
  Window ranges in ordered coordinates: 1:200, 201:400, 401:600, 601:1000
  Window sizes: 200, 200, 200, 400
Maxent initial obj = -700.7496748154989
Iter 1: obj = -592.9723427652139, δ = 0.3644471987139401, windows = 4
Iter 2: obj = -591.6845138262944, δ = 0.047909132919999875, windows = 4
Iter 3: obj = -591.6601051687553, δ = 0.004700979031753261, windows = 4
Iter 4: obj = -591.659661296202, δ = 0.0006610575234006211, windows = 4
Iter 5: obj = -591.6596530623799, δ = 8.819302816709396e-5, windows = 4
Iter 6: obj = -591.6596529269339, δ = 1.2030347537939079e-5, windows = 4
Iter 7: obj = -591.6596529223937, δ = 1.6264960241985094e-6, windows = 4
Post-optimization Cholesky check: passed.
  Verified L'L 

## Speed and FDR check

In [19]:
println("solve_s speedup: ", round(maxent_result.solve_time / fast_result.solve_time, digits=2), "x")
println("baseline FDP:    ", round(maxent_result.fdp, digits=3))
println("fast FDP:        ", round(fast_result.fdp, digits=3))
println("baseline power:  ", round(maxent_result.power, digits=3))
println("fast power:      ", round(fast_result.power, digits=3))
println("baseline min eig:", maxent_result.min_eig)
println("fast min eig:    ", fast_result.min_eig)

solve_s speedup: 2.85x
baseline FDP:    0.091
fast FDP:        0.091
baseline power:  1.0
fast power:      1.0
baseline min eig:0.24654608383770443
fast min eig:    0.24654605133874763


## Visualizing the 'Shadow column' and Early Termination

This section visualizes how a rank-1 update to a single coordinate `k` propagates through the Cholesky factor `L`.

When the update vector is sparse (`v = sqrt(delta)e_k`), the sequence of Givens rotations results in an updated `v` that is effectively a scaled copy of the `k`-th row of `L`. Due to the decay of correlations in genetic data, these updates eventually become numerically negligible, allowing for early termination.

In [ ]:
# Regenerate the manuscript version of Figure 1.
include("fig1.jl")